# Activity 3 Solution: Snowpark First Flight

This solution requires a live Snowflake connection, so it ships without stored outputs. Run the Step 0 SQL from the activity in Snowsight first; the checkpoints in each step are the expected results.

## Step 1: Connect and point at the table

In [ ]:
from snowflake.snowpark import Session
from configparser import ConfigParser

config = ConfigParser()
config.read("snow.cfg")
params = dict(config["DEV"])

session = Session.builder.configs(params).create()
session

In [ ]:
claims = session.table("W5D1_CLAIMS")
print(claims.count())  # 12
claims.show(12)

## Step 2: Totals per region

In [ ]:
from snowflake.snowpark.functions import col, sum as sum_

(
    claims
    .group_by("REGION")
    .agg(sum_("TOTAL_PAID").alias("REGION_TOTAL"))
    .sort("REGION")
    .show()
)
# East 368000, West 323000

## Step 3: Month-over-month change per region

In [ ]:
from snowflake.snowpark import Window
from snowflake.snowpark.functions import lag

w = Window.partition_by("REGION").order_by("CLAIM_MONTH")

enriched = (
    claims
    .with_column("PREV_TOTAL", lag("TOTAL_PAID").over(w))
    .with_column("MOM_CHANGE", col("TOTAL_PAID") - col("PREV_TOTAL"))
    .sort("REGION", "CLAIM_MONTH")
)
enriched.show(12)

## Step 4: Write it back

In [ ]:
enriched.write.mode("overwrite").save_as_table("W5D1_CLAIMS_ENRICHED")
session.table("W5D1_CLAIMS_ENRICHED").count()  # 12

## Step 5: Verify in Snowsight, then close

In [ ]:
session.close()

## Wrap up: sample answers

1. The first query runs at the first action. In this notebook that is `claims.count()` in Step 1 (`session.table` alone runs nothing).
2. `write_pandas`. The data lives on the laptop, so pandas reads it locally and `write_pandas` pushes it up in one call. Snowpark shines when the data is already in Snowflake.
3. `to_pandas` first would download all 50 million rows over the network into local memory just to reduce them to one number. Aggregate in Snowflake, then pull the single-row result.